[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C46_Graph_ML_Course/02_message_passing/02_message_passing.ipynb)

# 02 · 消息传递与 GCN（用 numpy 从零）

目标：从零实现 **MPNN 三件套**与 **GCN 一层 σ(ÂXW)**，对拍稀疏/稠密，训练一个 2 层 GCN 做节点分类（验证收敛），用 **Dirichlet 能量**量化过平滑。

路线：消息传递三件套 → GCN 归一化 Â（含自环） → 一层 GCN + 稀疏/稠密对拍 → 2 层 GCN 训练(SBM, 损失下降) → 过平滑(Dirichlet 能量) → ✏️ 练习 → 📖 答案 → 🧪 小 Cora-like 引文图胶囊。

> 心智模型：**一层 GCN = 模糊滤镜(ÂH, 无参数) + 调色(W, 学出来) + 非线性(σ)；层数 = 感受野半径，但太深会过平滑。**

## 1 · 消息传递三件套：消息 / 聚合 / 更新

MPNN：① 每个邻居发消息 `m=MSG(h_v,h_u)`；② 聚合 `a=AGG({m})`（**必须置换不变**：sum/mean/max）；③ 更新 `h'=UPD(h_v,a)`。
先把这三步写成显式的逐节点循环，建立「消息沿边流动」的具象。

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

def neighbors(A):
    return [np.nonzero(A[i])[0] for i in range(len(A))]

def message_passing(A, H, agg='sum'):
    '''最朴素 MPNN：消息=邻居特征本身；聚合=sum/mean/max；更新=直接用聚合结果。'''
    nbrs = neighbors(A)
    out = np.zeros_like(H)
    for v in range(len(A)):
        if len(nbrs[v]) == 0:
            continue
        msgs = H[nbrs[v]]                       # 每个邻居的消息
        if agg == 'sum':   a = msgs.sum(0)
        elif agg == 'mean': a = msgs.mean(0)
        elif agg == 'max':  a = msgs.max(0)
        out[v] = a                             # 更新=聚合结果
    return out

edges = [(0,1),(1,2),(0,2),(2,3),(3,4),(4,5),(3,5)]
n = 6; A = np.zeros((n,n))
for i,j in edges: A[i,j]=A[j,i]=1.0
H = np.array([[1.,0,0],[0,1.,0],[0,0,1.],[1.,1,0],[0,1.,1],[1.,0,1]])

agg_sum = message_passing(A, H, 'sum')
# sum 聚合等价于 A @ H（稠密对拍）
assert np.allclose(agg_sum, A @ H), 'sum 聚合应等于 A@H'
print('sum 聚合 == A @ H ✅')
print('mean 聚合:\n', message_passing(A, H, 'mean'))

**置换不变性验证**：聚合对邻居顺序不敏感。把某节点的邻居列表打乱，sum/mean/max 聚合结果不变。

In [ ]:
rng = np.random.default_rng(0)
v = 3                                  # 节点3 有 3 个邻居
nbrs_v = np.nonzero(A[v])[0]
for agg in ['sum','mean','max']:
    a1 = {'sum':H[nbrs_v].sum(0),'mean':H[nbrs_v].mean(0),'max':H[nbrs_v].max(0)}[agg]
    perm = rng.permutation(nbrs_v)
    a2 = {'sum':H[perm].sum(0),'mean':H[perm].mean(0),'max':H[perm].max(0)}[agg]
    assert np.allclose(a1, a2), f'{agg} 必须置换不变'
print('✅ sum/mean/max 聚合全部置换不变（打乱邻居顺序结果不变）')

## 2 · GCN 归一化 Â = D̃^{-1/2} Ã D̃^{-1/2}（加自环）

GCN 的核心：先 **加自环** Ã=A+I（否则节点忘掉自己），再 **对称归一化**。
验证：Â 对称、最大特征值≈1（重整化稳住谱）、每条边系数 = 1/√(d̃_i·d̃_j)。

In [ ]:
def gcn_norm(A):
    A_tilde = A + np.eye(len(A))           # 加自环
    d = A_tilde.sum(1)
    dinv_sqrt = np.diag(1.0 / np.sqrt(d))
    return dinv_sqrt @ A_tilde @ dinv_sqrt

A_hat = gcn_norm(A)
assert np.allclose(A_hat, A_hat.T), 'Â 必须对称'
lam_max = np.linalg.eigvalsh(A_hat).max()
print(f'Â 最大特征值 = {lam_max:.4f} (重整化把谱半径稳在 ~1)')
assert lam_max <= 1.0 + 1e-6, 'Â 谱半径应 ≤ 1'
# 手算核对一条边 (0,1) 的系数 = 1/sqrt(d̃_0 * d̃_1)
d_tilde = (A + np.eye(n)).sum(1)
assert np.allclose(A_hat[0,1], 1/np.sqrt(d_tilde[0]*d_tilde[1]))
print('✅ Â 对称、谱半径≈1、边系数=1/√(d̃_i d̃_j) 手算核对通过')

**有无自环的对比**：忘加自环 → 节点完全丢弃自身特征。我们对比验证这个常见 bug 的后果。

In [ ]:
def gcn_norm_no_selfloop(A):
    d = np.maximum(A.sum(1), 1e-12)
    dinv_sqrt = np.diag(1.0/np.sqrt(d))
    return dinv_sqrt @ A @ dinv_sqrt

A_hat_bad = gcn_norm_no_selfloop(A)
# 无自环：对角线全 0 -> 聚合时节点自身权重为 0（忘掉自己）
assert np.allclose(np.diag(A_hat_bad), 0), '无自环对角线为0'
assert np.all(np.diag(A_hat) > 0), '有自环对角线>0(保留自己)'
print('无自环 Â 对角线:', np.diag(A_hat_bad))
print('有自环 Â 对角线:', np.diag(A_hat).round(3))
print('✅ 加自环让节点在聚合中保留自身（对角线>0）——别忘了 A+I！')

## 3 · 一层 GCN：σ(ÂHW)，稀疏 vs 稠密对拍

一层 GCN = `relu(Â @ H @ W)`。真实框架走**稀疏**逐边聚合（不构造稠密 Â），我们写稀疏版与稠密版**对拍**。
再验证整层**置换等变**（重排节点，输出随之重排）。

In [ ]:
def relu(x): return np.maximum(x, 0.0)

def gcn_layer_dense(A_hat, H, W, act=relu):
    return act(A_hat @ H @ W)

def gcn_layer_sparse(A, H, W, act=relu):
    '''稀疏实现：逐边按 1/√(d̃_i d̃_j) 累加 + 自环，再线性变换。对拍稠密版。'''
    nA = len(A)
    d_tilde = (A + np.eye(nA)).sum(1)
    coef = 1.0 / np.sqrt(d_tilde)
    agg = np.zeros_like(H)
    for v in range(nA):
        agg[v] += coef[v]*coef[v] * H[v]       # 自环
    for i in range(nA):
        for j in range(nA):
            if A[i, j]:
                agg[i] += coef[i]*coef[j] * H[j]   # 真实边
    return act(agg @ W)

rng = np.random.default_rng(1)
W = rng.standard_normal((3, 4)) * 0.5
out_dense = gcn_layer_dense(A_hat, H, W)
out_sparse = gcn_layer_sparse(A, H, W)
assert np.allclose(out_dense, out_sparse), '稀疏与稠密 GCN 层必须一致'
print('✅ 稀疏(逐边) GCN == 稠密(ÂHW) GCN')

In [ ]:
# 置换等变：重排节点，输出随之重排
perm = rng.permutation(n)
P = np.eye(n)[perm]
A_perm = P @ A @ P.T
out_orig_perm = P @ gcn_layer_dense(gcn_norm(A), H, W)
out_perm = gcn_layer_dense(gcn_norm(A_perm), P @ H, W)
assert np.allclose(out_orig_perm, out_perm), 'GCN 层必须置换等变'
print('✅ GCN 层置换等变（先排后算 == 先算后排）')
# 结合律：ÂHW 两种乘法顺序一致
assert np.allclose((A_hat @ H) @ W, A_hat @ (H @ W))
print('✅ ÂHW 结合律：(ÂH)W == Â(HW)，框架据此择优')

## 4 · 训练一个 2 层 GCN 做节点分类（验证收敛）

从零实现完整训练：2 层 GCN + softmax 交叉熵 + 解析梯度 + 梯度下降。在 SBM(2 社区)上做半监督节点分类。
**正确性证据 = 损失单调下降 + 测试准确率高**。

In [ ]:
def make_sbm(sizes, p_in, p_out, seed=0):
    rng = np.random.default_rng(seed)
    nn = sum(sizes)
    labels = np.concatenate([np.full(s,k) for k,s in enumerate(sizes)])
    A = np.zeros((nn,nn))
    for i in range(nn):
        for j in range(i+1,nn):
            if rng.random() < (p_in if labels[i]==labels[j] else p_out):
                A[i,j]=A[j,i]=1.0
    return A, labels

def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z); return e / e.sum(1, keepdims=True)

Asbm, y = make_sbm([20,20], 0.4, 0.05, seed=2)
nn = len(Asbm); C = 2
Ah = gcn_norm(Asbm)
X = np.eye(nn)                          # 无信息特征：逼模型纯靠图结构
rng = np.random.default_rng(0)
train_mask = np.zeros(nn, bool)
for ccls in range(C):
    idx = np.where(y==ccls)[0]; train_mask[rng.choice(idx,5,replace=False)] = True
print(f'图 {nn} 节点，带标签 {train_mask.sum()} 个（每类5），其余测试')

In [ ]:
# 2 层 GCN: H1=relu(Ah X W0); Z=Ah H1 W1; P=softmax(Z)
rng = np.random.default_rng(3)
Fhid = 16
W0 = rng.standard_normal((nn, Fhid)) * 0.1
W1 = rng.standard_normal((Fhid, C)) * 0.1
lr = 0.5
AhX = Ah @ X                            # 预算一步
losses = []
for epoch in range(200):
    pre1 = AhX @ W0; H1 = relu(pre1)        # forward
    Z = Ah @ H1 @ W1; P = softmax(Z)
    logp = np.log(P[train_mask] + 1e-12)
    loss = -logp[np.arange(train_mask.sum()), y[train_mask]].mean()
    losses.append(loss)
    dZ = P.copy()                          # backward (只在 train 节点回传)
    dZ[train_mask, y[train_mask]] -= 1
    dZ[~train_mask] = 0
    dZ /= train_mask.sum()
    dW1 = H1.T @ (Ah @ dZ)
    dH1 = Ah @ dZ @ W1.T
    dpre1 = dH1 * (pre1 > 0)
    dW0 = AhX.T @ dpre1
    W0 -= lr * dW0; W1 -= lr * dW1
pred = P.argmax(1)
test_acc = (pred[~train_mask] == y[~train_mask]).mean()
print(f'初始 loss={losses[0]:.4f} -> 末 loss={losses[-1]:.4f}')
print(f'测试准确率 = {test_acc:.2%}')
assert losses[-1] < losses[0] * 0.5, '损失应显著下降'
assert test_acc >= 0.85, '2层GCN应能用图结构+少量标签分对大部分节点'
print('✅ 损失显著下降、测试准确率高 —— 2 层 GCN 从零训练成功')

## 5 · 过平滑：Dirichlet 能量随层数指数衰减

反复用 Â 聚合 = 反复扩散 → 节点表示趋同（**过平滑**）。
用 **Dirichlet 能量** `E(H)=tr(Hᵀ L H)=½Σ‖h_i−h_j‖²` 量化：过平滑 ⟺ E 随层数指数衰减到 0。

In [ ]:
def dirichlet_energy(A, H):
    L = np.diag(A.sum(1)) - A
    return float(np.trace(H.T @ L @ H))

Ah = gcn_norm(Asbm)
Hr = rng.standard_normal((nn, 8))      # 随机信号(注意不复用 6 节点的 H)
energies = [dirichlet_energy(Asbm, Hr)]
Hl = Hr.copy()
for l in range(1, 21):
    Hl = Ah @ Hl                         # 一步传播
    energies.append(dirichlet_energy(Asbm, Hl))
print('Dirichlet 能量随层数：')
for l in [0,1,2,5,10,20]:
    print(f'  层 {l:2d}: E = {energies[l]:.4e}')
assert energies[20] < energies[0] * 1e-2, '深层应过平滑（能量趋0）'
assert all(energies[l] >= energies[l+1] - 1e-9 for l in range(20)), '能量应单调不增'
print('✅ 能量单调衰减到 ~0 —— 这就是过平滑：层数越深节点越无法区分')

**直观确认**：过平滑后所有节点表示几乎相同——任意两节点表示的差异范数趋于 0。

In [ ]:
Hdeep = Hr.copy()
for _ in range(50): Hdeep = Ah @ Hdeep
Hdeep = Hdeep / (np.linalg.norm(Hdeep, axis=1, keepdims=True) + 1e-12)
pairwise = np.linalg.norm(Hdeep[0] - Hdeep[1])
print(f'50 层后节点0与节点1 表示(归一化)的差异 = {pairwise:.2e}')
assert pairwise < 0.1, '深层后节点表示趋同'
print('✅ 过平滑确认：深层后节点表示几乎无法区分')

---
## ✏️ 练习 1：mean 聚合 = 行归一化传播

实现 `mean_aggregate(A, H)`：每个节点取邻居特征的**平均**（不含自己）。
验证它等于行归一化传播 `D^{-1} A H`（D 为度对角阵）。孤立点（度0）输出全 0。

In [ ]:
def mean_aggregate(A, H):
    # TODO: deg=A.sum(1); 对每个节点输出邻居特征均值；
    #       等价于 diag(1/max(deg,1)) @ A @ H（度0的行结果置0）
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
Hm = mean_aggregate(A, H)
deg = A.sum(1)
ref = np.diag(1.0/np.maximum(deg,1)) @ A @ H
assert np.allclose(Hm, ref), 'mean 聚合应等于 D^{-1} A H'
assert np.allclose(Hm[0], (H[1]+H[2])/2)   # 节点0邻居{1,2}
print('✅ 练习 1 通过：mean 聚合 == 行归一化传播 D^{-1}AH')

## ✏️ 练习 2：GCN 传播规则（多层，无参数版）

实现 `gcn_propagate(A, X, k)`：返回 `Â^k X`（k 步 GCN 传播，不含 W/σ）。
这是 SGC（Simplifying GCN）的核心——预计算传播。验证 k 步等于逐步传播 k 次。

In [ ]:
def gcn_propagate(A, X, k):
    # TODO: Ah = gcn_norm(A); 反复左乘 k 次 -> Ah^k X
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
Xt = rng.standard_normal((n, 3))
out_k = gcn_propagate(A, Xt, 3)
Ah_ = gcn_norm(A)
ref_k = Ah_ @ (Ah_ @ (Ah_ @ Xt))
assert np.allclose(out_k, ref_k), 'Â^3 X 应等于逐步传播3次'
assert np.allclose(gcn_propagate(A, Xt, 0), Xt), 'k=0 应返回 X 本身'
print('✅ 练习 2 通过：GCN 多步传播 Â^k X（SGC 的预计算核心）')

## ✏️ 练习 3：过平滑度量

实现 `oversmoothing_ratio(A, X, k)`：返回 `E(Â^k X) / E(X)`，即 k 层传播后 Dirichlet 能量相对初始的比值。
比值越接近 0 = 过平滑越严重。验证：k 越大比值越小（单调）。

In [ ]:
def oversmoothing_ratio(A, X, k):
    # TODO: 用 gcn_propagate 得 Â^k X；用 dirichlet_energy 算能量比
    #       注意 E(X) 可能很小，可加 1e-12 防除零
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
Xs = rng.standard_normal((nn, 8))
r1 = oversmoothing_ratio(Asbm, Xs, 1)
r10 = oversmoothing_ratio(Asbm, Xs, 10)
assert 0 <= r10 <= r1 <= 1.0 + 1e-6, '层数越深能量比越小'
assert r10 < 0.1, '10 层后应明显过平滑'
print(f'能量比: 1层={r1:.3f}, 10层={r10:.4f}')
print('✅ 练习 3 通过：过平滑度量随层数单调下降')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def mean_aggregate(A, H):
    deg = A.sum(1)
    return np.diag(1.0/np.maximum(deg,1)) @ A @ H

In [ ]:
# 练习 2 参考答案
def gcn_propagate(A, X, k):
    Ah = gcn_norm(A)
    out = X.copy()
    for _ in range(k):
        out = Ah @ out
    return out

In [ ]:
# 练习 3 参考答案
def oversmoothing_ratio(A, X, k):
    Xk = gcn_propagate(A, X, k)
    return dirichlet_energy(A, Xk) / (dirichlet_energy(A, X) + 1e-12)

---
## 🧪 真实数据胶囊：小 Cora-like 引文图节点分类

Cora 是引文网络的经典 benchmark：节点=论文、边=引用、标签=主题，**强同配**（互引的论文多半同主题）——GCN 的理想战场。
我们优先尝试取真实 Cora（若环境有缓存），失败则**生成一个统计上仿 Cora 的同配引文图**（带主题标签，结构真实）。

In [ ]:
def load_cora_like(seed=0):
    '''优先真实 Cora；失败回退到仿 Cora 的同配引文图(3 主题, 强同配)。'''
    try:
        raise ImportError   # 离线环境无真实 Cora，直接走回退
    except Exception:
        rng = np.random.default_rng(seed)
        sizes = [40, 40, 40]                # 3 个主题
        A, y = make_sbm(sizes, p_in=0.18, p_out=0.012, seed=seed)  # 强同配
        nn = len(A)
        F = 16
        protos = rng.standard_normal((3, F))   # 主题原型
        X = np.array([protos[y[i]] for i in range(nn)]) + 0.6*rng.standard_normal((nn,F))
        return A, X, y, 'synthetic-cora'

A_c, X_c, y_c, src = load_cora_like(seed=1)
print(f'引文图来源: {src}, {len(A_c)} 节点, {int(A_c.sum()//2)} 边, {len(set(y_c.tolist()))} 主题')
same = [y_c[i]==y_c[j] for i in range(len(A_c)) for j in range(i+1,len(A_c)) if A_c[i,j]]
homophily = np.mean(same)
print(f'同配率(edge homophily) = {homophily:.2%}')
assert homophily > 0.7, '引文图应强同配'
print('✅ 引文图就绪，强同配 —— GCN 的理想场景')

**🧪 胶囊练习**：用 `gcn_propagate`（练习 2）做 **SGC 式分类**：预计算 `Â^2 X_c`，在其上训一个**线性 softmax** 分类器（仅传播+线性，无隐层），
对每类用 10 个标签训练，报告测试准确率。提示：传播已注入图结构，线性分类器就够了。

In [ ]:
# 你的任务：补全 Xp（预计算传播）这一行。
def sgc_classify(A, X, y, k=2, n_label=10, seed=0, epochs=300, lr=0.3):
    rng = np.random.default_rng(seed)
    nn = len(A); C = len(set(y.tolist()))
    Xp = None      # TODO: gcn_propagate(A, X, k)
    train = np.zeros(nn, bool)
    for cc in range(C):
        idx = np.where(y==cc)[0]; train[rng.choice(idx,n_label,replace=False)]=True
    W = rng.standard_normal((Xp.shape[1], C))*0.1; b = np.zeros(C)
    for _ in range(epochs):
        Z = Xp @ W + b; P = softmax(Z)
        dZ = P.copy(); dZ[train, y[train]] -= 1; dZ[~train]=0; dZ/=train.sum()
        W -= lr * (Xp.T @ dZ); b -= lr * dZ.sum(0)
    pred = (Xp @ W + b).argmax(1)
    return (pred[~train]==y[~train]).mean()

raise NotImplementedError  # 删除本行并补全上面的 Xp

In [ ]:
# 自测
acc = sgc_classify(A_c, X_c, y_c, k=2)
print(f'SGC(Â²X + 线性) 测试准确率 = {acc:.2%}')
assert acc >= 0.8, 'SGC 在强同配引文图上应表现好'
print('✅ 胶囊练习通过：仅「传播+线性」就能在引文图上分对大多数论文')

In [ ]:
# 📖 胶囊参考答案
def sgc_classify(A, X, y, k=2, n_label=10, seed=0, epochs=300, lr=0.3):
    rng = np.random.default_rng(seed)
    nn = len(A); C = len(set(y.tolist()))
    Xp = gcn_propagate(A, X, k)            # 预计算 Â^k X
    train = np.zeros(nn, bool)
    for cc in range(C):
        idx = np.where(y==cc)[0]; train[rng.choice(idx,n_label,replace=False)]=True
    W = rng.standard_normal((Xp.shape[1], C))*0.1; b = np.zeros(C)
    for _ in range(epochs):
        Z = Xp @ W + b; P = softmax(Z)
        dZ = P.copy(); dZ[train, y[train]] -= 1; dZ[~train]=0; dZ/=train.sum()
        W -= lr*(Xp.T @ dZ); b -= lr*dZ.sum(0)
    pred = (Xp @ W + b).argmax(1)
    return (pred[~train]==y[~train]).mean()

acc = sgc_classify(A_c, X_c, y_c, k=2)
print(f'SGC 测试准确率 = {acc:.2%}')
assert acc >= 0.8
print('✅ 传播(注入图结构) + 线性分类器，已足够分对大多数论文')

### 小结
- **MPNN 三件套**：消息→聚合(置换不变 sum/mean/max)→更新，统一一切空间型 GNN。
- **GCN 一层 = σ(ÂHW)**：Â=D̃^{-1/2}ÃD̃^{-1/2}(加自环+对称归一化)；模糊滤镜(ÂH)+调色(W)+非线性。
- **稀疏==稠密、置换等变、结合律** 全部对拍通过；2 层 GCN 半监督训练损失单调下降。
- **过平滑**：反复 Â 聚合=扩散收敛 → Dirichlet 能量指数衰减到 0 → 层数≠越深越好。

下一站：**模块 03 · GAT 与 GraphSAGE** —— 让聚合权重可学(GAT)、让模型可采样可归纳(SAGE)，修补 GCN 的两条局限。